# 01-Agentic-RAG Assignment

# Load the data 

In [3]:
from minsearch import Index

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [5]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Q1. How many lesson pages

In [4]:
print(fr"Number of lessons: {len(documents)}")

Number of lessons: 72


In [5]:
documents[0].keys()

dict_keys(['content', 'filename'])

## Q2. Indexing and searching

Index the documents with minsearch - make `content` a text field and
`filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

* `01-agentic-rag/lessons/03-rag.md`
* `01-agentic-rag/lessons/14-agentic-loop.md`
* `04-evaluation/lessons/13-llm-as-judge.md`
* `06-best-practices/lessons/02-hybrid-search.md`

In [6]:
query = "How does the agentic loop keep calling the model until it stops?"
index = Index(text_fields=['content'], keyword_fields=['filename'])
index.fit(documents)
search_result_Q2 = index.search(query, num_results=1)
print(fr"First result's filename is {search_result_Q2[0]['filename']}")

First result's filename is 01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper 
script we prepared during the lessons:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`),
while our documents have `filename` and `content`.

Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for
this request?

* 700
* 7000
* 70000
* 700000

We count input tokens instead of price because the cost depends on the model
and provider you use, but the size of the prompt we send is the same for
everyone.

Most LLM APIs report token usage on the response object (e.g.
`response.usage.input_tokens` / `prompt_tokens`). We'll read the input tokens
from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, `llm` returns only the text. Modify it to return the whole response, and change `rag` to return both the answer and usage (as a tuple or create a small dataclass for that).

In [7]:
from rag_helper import LessonRAG
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [8]:
lesson_rag = LessonRAG(index = index, llm_client=openai_client)

In [ ]:
response = lesson_rag.rag(query)

In [ ]:
print(fr"input (prompt) tokens did we send to the model for this request is {response.input_tokens}")

input (prompt) tokens did we send to the model for this request is 7121


## Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents
make retrieval less precise: a match deep inside a page still pulls in the
whole page. A common fix is chunking: split each page into smaller,
overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding
window - a window of `size` characters slides across the text in steps of
`step` characters, and each window position becomes one chunk:

```python
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

With `size=2000` and `step=1000` (you can see the implementation
[here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step`
  is smaller than `size`, consecutive chunks overlap by `size - step` (1000)
  characters, so a passage split across a boundary still appears whole in one
  of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the
  offset in the page) and `content` (the chunk text).

How many chunks do you get?

* 70
* 295
* 1100
* 4500

In [13]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(fr'Number of chunks: {len(chunks)}')

Number of chunks: 295


## Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the
LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field,
`filename` as a keyword field), point your RAG at the chunk index, and
answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked
version send?

* about the same
* 3× fewer
* 10× fewer
* 30× fewer

In [ ]:
chunk_index = Index(text_fields=['content'], keyword_fields=['filename'])
chunk_index.fit(chunks)
lesson_chunk_rag = LessonRAG(index = chunk_index, llm_client=openai_client)
chunk_response = lesson_chunk_rag.rag(query)
print(fr"input (prompt) tokens did we send to the model for this request is {chunk_response.input_tokens}")

input (prompt) tokens did we send to the model for this request is 2304


## Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give
the LLM a `search` tool and let it decide when (and what) to search. We
suggest [toyaikit](https://github.com/alexeygrigorev/toyaikit), the small
agent library from the module, but you can use anything you like - the OpenAI
Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

```bash
uv add toyaikit
```

Create a `search` function that uses the chunk index. Give it a type hint and
a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your `search` tool and run it (with toyaikit, the same way
as in the ToyAIKit lesson). Use these instructions for the agent (they nudge
it to search a few times):

> You're a course teaching assistant. Answer the student's question using the
> search tool. Make multiple searches with different keywords before answering.

Ask it:

> How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer. Count how many
times it called the `search` tool.

How many times did the agent call `search`?

> Note: the agent decides this itself, so it varies a little between runs -
> pick the closest option. We measured this with OpenAI `gpt-5.4-mini`; with a
> different model or provider the number may differ, so keep that in mind.

* 0
* 4
* 10
* 20 

In [ ]:
from langchain_rag_helper import LangChainAgenticRAG
from minsearch import Index
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(fr'Number of chunks: {len(chunks)}')
chunk_index = Index(text_fields=['content'], keyword_fields=['filename'])
chunk_index.fit(chunks)
agentic_rag = LangChainAgenticRAG(
    index=chunk_index,
    model="gpt-5-mini",
    num_results=5,
)

response = agentic_rag.rag(
    "How does the agentic loop work, and how is it different from plain RAG?"
)

print(response.answer)

print("\nSearch calls:", response.search_call_count)

Number of chunks: 295
Short answer
- Plain RAG: one retrieval + one LLM call. Search (lexical or vector) → build prompt using the retrieved context → single LLM call that returns the answer.
- Agentic loop: the LLM is put in the driver's seat and can call tools (search, etc.) repeatedly inside a loop: the system sends messages, the model may return a function/tool call, you run that tool, append the tool output to the conversation, then call the model again — repeat until the model returns a final answer with no tool calls.

How the agentic loop works (from the course)
- Components:
  - Instructions: a developer/system message that defines the agent’s role and behavior (e.g., “use the search function, make multiple searches, expand keywords...”).
  - Tools: functions the agent can call (in the lessons the search tool is the example).
  - Memory: the full message history (every prompt, model output, and tool result) that you resend to the LLM each round.
- The loop:
  1. Send user + dev